# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Pipeline 1 (Baseline):** Bi-Encoder embeddings -> Top 10 by cosine similarity -> Top 3 to LLM -> Answer 1  
**Pipeline 2 (Re-Ranking):** same Top 10 -> Cross-Encoder re-scores (query, chunk) pairs -> Top 3 -> LLM (same prompt) -> Answer 2

Sections: 0 Setup, 1 Load + chunk PDF, 2 Index, 3 Pipelines, 4 Failure-mode stress tests, 5 Rank shifts, 6 Answer comparison, 7 Latency, 8 Scaling, 9 Report notes.

## 0. Setup & Environment

In [1]:
# Run once (Colab / Jupyter). Remove "-q" to see install logs.
!pip install -q pypdf sentence-transformers faiss-cpu google-genai pandas numpy python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 94.7 MB/s eta 0:00:00


In [2]:
import os, re, time, textwrap
import numpy as np
import pandas as pd
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

# ------------------------- CONFIG (edit these) -------------------------
PDF_PATH = "charter.pdf"
CHUNK_WORDS = 110                  # words per chunk
CHUNK_OVERLAP = 25                 # overlapping words between consecutive chunks
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K_RETRIEVE = 10                # candidates from the bi-encoder
TOP_K_FINAL = 3                    # chunks sent to the LLM
LLM_MODEL = "gemini-2.5-flash"
# -----------------------------------------------------------------------

# Load Gemini keys API_1 ... API_6 from env.txt
from dotenv import load_dotenv
load_dotenv("env.txt")
API_KEYS = [os.environ[f"API_{i}"] for i in range(1, 7) if os.environ.get(f"API_{i}")]
assert API_KEYS, "No API_1..API_6 keys found in env.txt"
print(f"Loaded {len(API_KEYS)} Gemini API keys")

Loaded 6 Gemini API keys


In [3]:
# Colab only: upload the PDF if it is not already present.
if not os.path.exists(PDF_PATH):
    try:
        from google.colab import files
        up = files.upload()
        PDF_PATH = list(up.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"Put your PDF at {PDF_PATH} or change PDF_PATH.")
print("Using PDF:", PDF_PATH)

Using PDF: charter.pdf


## 1. Load the PDF and chunk it
Both pipelines use the **exact same** chunk list.

In [4]:
def load_pdf_pages(path):
    reader = PdfReader(path)
    pages = []
    for i, p in enumerate(reader.pages, start=1):
        txt = p.extract_text() or ""
        txt = re.sub(r"\s+", " ", txt).strip()      # normalise whitespace
        if txt:
            pages.append((i, txt))
    return pages

def chunk_pages(pages, chunk_words=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    # Sliding word-window over the whole document; each chunk remembers the page it starts on.
    words, word_page = [], []
    for pg, txt in pages:
        w = txt.split()
        words.extend(w)
        word_page.extend([pg] * len(w))
    chunks, step = [], chunk_words - overlap
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_words]
        if len(piece) < 30 and chunks:               # avoid tiny trailing chunk
            break
        chunks.append({"id": len(chunks), "page": word_page[start], "text": " ".join(piece)})
    return chunks

pages = load_pdf_pages(PDF_PATH)
chunks = chunk_pages(pages)
print(f"Pages with text: {len(pages)} | Chunks: {len(chunks)}")
print("\nSample chunk:\n", textwrap.fill(chunks[0]["text"][:500], 100))

Pages with text: 38 | Chunks: 123

Sample chunk:
 CHARTER OF THE STUDENTS’ PARLIAMENT OF IIIT HYDERABAD parliament.iiit.ac.in/charter CONTENTS
Preamble 1 Part I General Provisions 3 Article I Identity . . . . . . . . . . . . . . . . . . . . .
3 Part II Structure 5 Article I Composition . . . . . . . . . . . . . . . . . . . 5 Article II Terms
of Office . . . . . . . . . . . . . . . . 6 Article III Councils . . . . . . . . . . . . . . . .


## 2. Dense index (Bi-Encoder + FAISS inner product on normalised vectors = cosine similarity)

In [5]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

t0 = time.perf_counter()
chunk_texts = [c["text"] for c in chunks]
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=64, normalize_embeddings=True,
                              show_progress_bar=True).astype("float32")
index_build_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_emb.shape[1])   # inner product == cosine because vectors are L2-normalised
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]}, build time={index_build_time:.2f}s")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Indexed 123 chunks, dim=384, build time=1.63s


## 3. The two pipelines

In [6]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)

def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """Pipeline 1, steps a-b: embed query, return Top-k (chunk_id, cosine score)."""
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

def rerank(query, candidates, k=TOP_K_FINAL):
    """Pipeline 2, steps a-c: score all (query, chunk) pairs jointly, sort descending."""
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)
    ranked = sorted(zip([cid for cid, _ in candidates], ce_scores), key=lambda x: x[1], reverse=True)
    return [(int(cid), float(s)) for cid, s in ranked], [(int(cid), float(s)) for cid, s in ranked[:k]]

PROMPT_TEMPLATE = """You are answering questions about a document using ONLY the context below.
If the context does not contain the answer, say "The provided context does not contain the answer."
Mention any exceptions or restrictions that appear in the context.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(query, chunk_ids):
    ctx = "\n\n".join(f"[Chunk {cid} | page {chunks[cid]['page']}]\n{chunks[cid]['text']}" for cid in chunk_ids)
    return PROMPT_TEMPLATE.format(context=ctx, question=query)

from google import genai

_key_idx = 0   # index of the key currently in use

def generate_answer(prompt, max_cycles=3):
    """Use the current key; on any error jump to the next key, wrapping around the list."""
    global _key_idx
    for attempt in range(max_cycles * len(API_KEYS)):
        try:
            client = genai.Client(api_key=API_KEYS[_key_idx])
            return client.models.generate_content(model=LLM_MODEL, contents=prompt).text.strip()
        except Exception as e:
            print(f"  [key #{_key_idx + 1} failed: {str(e)[:80]}] -> switching key")
            _key_idx = (_key_idx + 1) % len(API_KEYS)
            if (attempt + 1) % len(API_KEYS) == 0:
                time.sleep(5)          # all keys failed once; brief pause before next round
    return "[LLM error: all keys failed]"

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [7]:
def run_both(query, gold_phrase=None, call_llm=True, verbose=True):
    """Runs Pipeline 1 and Pipeline 2 on one query and returns a result dict with timings."""
    # ---- Pipeline 1: baseline ----
    t0 = time.perf_counter()
    top10 = baseline_retrieve(query)
    t_base = time.perf_counter() - t0
    base_top3 = [cid for cid, _ in top10[:TOP_K_FINAL]]

    # ---- Pipeline 2: re-rank the same top 10 ----
    t0 = time.perf_counter()
    full_rerank, rr_top3 = rerank(query, top10)
    t_rerank = time.perf_counter() - t0
    rr_top3_ids = [cid for cid, _ in rr_top3]

    # ---- Rank bookkeeping ----
    base_rank = {cid: r + 1 for r, (cid, _) in enumerate(top10)}
    rr_rank = {cid: r + 1 for r, (cid, _) in enumerate(full_rerank)}

    # ---- Locate the "vital" chunk via a gold phrase (a distinctive string from the true answer) ----
    gold_ids = []
    if gold_phrase:
        gold_ids = [cid for cid, _ in top10 if gold_phrase.lower() in chunks[cid]["text"].lower()]
    gold_info = [(cid, base_rank[cid], rr_rank[cid]) for cid in gold_ids]

    ans1 = ans2 = None
    t_llm1 = t_llm2 = 0.0
    if call_llm:
        t0 = time.perf_counter(); ans1 = generate_answer(build_prompt(query, base_top3)); t_llm1 = time.perf_counter() - t0
        t0 = time.perf_counter(); ans2 = generate_answer(build_prompt(query, rr_top3_ids)); t_llm2 = time.perf_counter() - t0

    res = dict(query=query, top10=top10, full_rerank=full_rerank, base_top3=base_top3,
               rr_top3=rr_top3_ids, gold=gold_info, ans1=ans1, ans2=ans2,
               t_base=t_base, t_rerank=t_rerank, t_llm1=t_llm1, t_llm2=t_llm2)
    if verbose:
        print("="*100); print("QUERY:", query)
        print("\nBaseline Top10 (chunk_id: cosine) ->", [(c, round(s, 3)) for c, s in top10])
        print("Re-ranked order (chunk_id: CE score) ->", [(c, round(s, 2)) for c, s in full_rerank])
        print("Baseline Top3:", base_top3, "| Re-ranked Top3:", rr_top3_ids)
        for cid, b, r in gold_info:
            print(f"  >> Vital chunk {cid}: Rank #{b} -> Rank #{r}")
        if not gold_info and gold_phrase:
            print("  >> Gold phrase NOT found in the baseline top 10 (a retrieval-recall miss; try a different phrase/query).")
        if call_llm:
            print("\n--- Answer 1 (baseline) ---\n", textwrap.fill(ans1, 110))
            print("\n--- Answer 2 (re-ranked) ---\n", textwrap.fill(ans2, 110))
    return res

## 4. Failure-mode stress tests
Candidate queries written from the charter text. `gold_phrase` is a short string from the chunk that truly answers the query; the notebook uses it to track that chunk's rank before and after re-ranking.  
Step 1 screens all candidates (retrieval only, no LLM calls). Step 2 keeps the ones where the vital chunk is outside the baseline Top 3 but inside the re-ranked Top 3 and runs the full pipelines (with the LLM) on them.

In [8]:
candidates = [
    # --- Keyword trap / synonyms ---
    {"category": "Keyword Trap / Distractor",
     "query": "What happens to the Parliament if nobody ends up as General Secretary even after a second attempt?",
     "gold_phrase": "deemed dissolved with immediate effect"},
    {"category": "Keyword Trap / Distractor",
     "query": "How many Councils can a Member of Parliament join?",
     "gold_phrase": "part of no more than four Councils"},
    # --- Exceptions & corner cases ---
    {"category": "Exceptions & Corner Cases",
     "query": "Does the Speaker have a vote inside Councils?",
     "gold_phrase": "breaking a tie"},
    {"category": "Exceptions & Corner Cases",
     "query": "What is the limit on how many Nominated Members a Council can have?",
     "gold_phrase": "this cap may be exceeded"},
    # --- Negation & constraints ---
    {"category": "Negation & Constraints",
     "query": "Which positions is the General Secretary not allowed to hold?",
     "gold_phrase": "The General Secretary shall not hold any of the following positions"},
    {"category": "Negation & Constraints",
     "query": "Do Nominated Members get to vote in full Parliament sessions?",
     "gold_phrase": "possess no vote in general Sessions"},
    # --- Other (vital chunk buried among near-duplicates) ---
    {"category": "Other (buried at rank 4-10)",
     "query": "How quickly must Parliament acknowledge a petition from a student?",
     "gold_phrase": "within three days of receiving the communication"},
    {"category": "Other (buried at rank 4-10)",
     "query": "How long can the General Secretary speak in their own defence at a dismissal hearing?",
     "gold_phrase": "entitled to exactly fifteen minutes"},
    {"category": "Other (buried at rank 4-10)",
     "query": "How often must a Member of Parliament hold office hours?",
     "gold_phrase": "at least once every fifteen days"},
]

In [9]:
# Step 1: screening (retrieval + re-ranking only, no LLM). USEFUL = vital chunk outside baseline Top 3, inside re-ranked Top 3.
rows = []
for i, c in enumerate(candidates):
    r = run_both(c["query"], gold_phrase=c["gold_phrase"], call_llm=False, verbose=False)
    if r["gold"]:
        b = min(g[1] for g in r["gold"]); rr = min(g[2] for g in r["gold"])
    else:
        b = rr = None
    rows.append({"idx": i, "category": c["category"], "query": c["query"][:60],
                 "baseline rank": b, "reranked rank": rr,
                 "USEFUL": bool(b is not None and b > TOP_K_FINAL and rr <= TOP_K_FINAL)})
screen_df = pd.DataFrame(rows)
screen_df

,idx,category,query,baseline rank,reranked rank,USEFUL
0,0,Keyword Trap / Distractor,What happens to the Parliament if nobody ends ...,1.0,1.0,False
1,1,Keyword Trap / Distractor,How many Councils can a Member of Parliament j...,1.0,1.0,False
2,2,Exceptions & Corner Cases,Does the Speaker have a vote inside Councils?,3.0,2.0,False
3,3,Exceptions & Corner Cases,What is the limit on how many Nominated Member...,6.0,2.0,True
4,4,Negation & Constraints,Which positions is the General Secretary not a...,NaN,NaN,False
5,5,Negation & Constraints,Do Nominated Members get to vote in full Parli...,1.0,1.0,False
6,6,Other (buried at rank 4-10),How quickly must Parliament acknowledge a peti...,NaN,NaN,False
7,7,Other (buried at rank 4-10),How long can the General Secretary speak in th...,7.0,2.0,True
8,8,Other (buried at rank 4-10),How often must a Member of Parliament hold off...,1.0,1.0,False


In [10]:
# Step 2: choose the queries for the full run. Set KEEP_OVERRIDE = [idx, ...] to pick manually, else all USEFUL ones are used.
KEEP_OVERRIDE = None
KEEP = KEEP_OVERRIDE if KEEP_OVERRIDE is not None else [int(i) for i in screen_df.loc[screen_df["USEFUL"], "idx"]]
if len(KEEP) < 2:
    print("WARNING: fewer than 2 candidates show the failure. Using all candidates that were found in the Top 10; reword queries if needed.")
    KEEP = [int(i) for i in screen_df.loc[screen_df["baseline rank"].notna(), "idx"]] or list(range(len(candidates)))
failure_queries = [candidates[i] for i in KEEP]
print("Selected candidates:", KEEP)

Selected candidates: [3, 7]


In [11]:
results = []
for fq in failure_queries:
    if fq["query"].startswith("REPLACE"):
        print("Skipping placeholder:", fq["category"]); continue
    print("\n\n######", fq["category"])
    r = run_both(fq["query"], gold_phrase=fq["gold_phrase"])
    r["category"] = fq["category"]
    results.append(r)



###### Exceptions & Corner Cases
QUERY: What is the limit on how many Nominated Members a Council can have?

Baseline Top10 (chunk_id: cosine) -> [(14, 0.676), (15, 0.637), (13, 0.627), (29, 0.613), (45, 0.592), (16, 0.587), (49, 0.553), (18, 0.548), (55, 0.505), (36, 0.485)]
Re-ranked order (chunk_id: CE score) -> [(14, 2.12), (16, 1.89), (13, 0.51), (15, -0.75), (49, -3.4), (45, -3.44), (55, -3.58), (29, -4.15), (18, -9.46), (36, -10.07)]
Baseline Top3: [14, 15, 13] | Re-ranked Top3: [14, 16, 13]
  >> Vital chunk 16: Rank #6 -> Rank #2

--- Answer 1 (baseline) ---
 The number of Nominated Members appointed to a Council shall not exceed one-half of the number of elected
members.

--- Answer 2 (re-ranked) ---
 The number of Nominated Members appointed to a Council shall not exceed one-half of the number of elected
Members of Parliament appointed to that Council.  **Exception:** This cap may be exceeded if and only if
necessary to meet the composition requirements of Part IV.


######

## 5. Rank-shift summary (vital chunk: baseline rank -> re-ranked rank)

In [12]:
rows = []
for r in results:
    if r["gold"]:
        for cid, b, rr in r["gold"]:
            rows.append({"Category": r["category"], "Query": r["query"][:70], "Vital chunk": cid,
                         "Baseline rank": b, "Re-ranked rank": rr, "Change": b - rr,
                         "In baseline Top3?": b <= TOP_K_FINAL, "In re-ranked Top3?": rr <= TOP_K_FINAL})
    else:
        rows.append({"Category": r["category"], "Query": r["query"][:70], "Vital chunk": None,
                     "Baseline rank": None, "Re-ranked rank": None, "Change": None,
                     "In baseline Top3?": False, "In re-ranked Top3?": False})
rank_df = pd.DataFrame(rows)
rank_df

,Category,Query,Vital chunk,Baseline rank,Re-ranked rank,Change,In baseline Top3?,In re-ranked Top3?
0,Exceptions & Corner Cases,What is the limit on how many Nominated Member...,16,6,2,4,False,True
1,Other (buried at rank 4-10),How long can the General Secretary speak in th...,84,7,2,5,False,True


## 6. Answer 1 vs. Answer 2
Read the printed answers above and, for each failure query, note in your own words whether Answer 1 hallucinated, omitted a clause, or misstated a rule, and whether Answer 2 fixed it. The cell below prints them side by side with the chunks used so you can inspect the evidence.

In [13]:
def show_chunks(ids, label):
    print(f"\n{label}")
    for cid in ids:
        print(f"  [Chunk {cid} | p.{chunks[cid]['page']}] {textwrap.shorten(chunks[cid]['text'], 220)}")

for r in results:
    print("\n" + "#"*100); print(r["category"], "|", r["query"])
    show_chunks(r["base_top3"], "Baseline Top 3 chunks:")
    show_chunks(r["rr_top3"], "Re-ranked Top 3 chunks:")
    print("\nAnswer 1:", textwrap.fill(r["ans1"], 110))
    print("\nAnswer 2:", textwrap.fill(r["ans2"], 110))


####################################################################################################
Exceptions & Corner Cases | What is the limit on how many Nominated Members a Council can have?

Baseline Top 3 chunks:
  [Chunk 14 | p.9] of Parliament shall be part of no more than four Councils. Each Member shall choose three Councils to join. (d) If the requirements for the composition of a Council are not met, the General Secretary may assign [...]
  [Chunk 15 | p.9] if required. (c) The General Secretary, the Speaker, and the Members of the respec- tive Council or Subcouncil shall interview all candidates for that Council or Subcouncil. 5 (d) The Speaker shall provide the [...]
  [Chunk 13 | p.9] election, or to otherwise assist a Council or Subcouncil in its duties, shall be defined as a Nominated Member of a Council or Subcouncil. Nominated Members hold voting rights strictly within their assigned Council [...]

Re-ranked Top 3 chunks:
  [Chunk 14 | p.9] of Parliament shall be 

## 7. Latency profiling
Retrieval-only timings (the LLM call is identical in both pipelines, so it is reported separately). Each query is repeated to average out noise; the first call is a warm-up.

In [14]:
def time_pipeline(query, repeats=5):
    baseline_retrieve(query); rerank(query, baseline_retrieve(query))   # warm-up
    tb, tr = [], []
    for _ in range(repeats):
        t0 = time.perf_counter(); top10 = baseline_retrieve(query); tb.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); rerank(query, top10); tr.append(time.perf_counter() - t0)
    return np.mean(tb), np.mean(tr)

lat_rows = []
for r in results:
    mb, mr = time_pipeline(r["query"])
    lat_rows.append({"Query": r["query"][:60],
                     "Baseline retrieval (ms)": mb*1000,
                     "Re-rank 10 pairs (ms)": mr*1000,
                     "Pipeline 2 total (ms)": (mb+mr)*1000,
                     "LLM call P1 (s)": r["t_llm1"], "LLM call P2 (s)": r["t_llm2"]})
lat_df = pd.DataFrame(lat_rows).round(2)
if len(lat_df):
    avg = lat_df.drop(columns="Query").mean().round(2)
    lat_df.loc[len(lat_df)] = ["AVERAGE"] + list(avg.values)
lat_df

,Query,Baseline retrieval (ms),Re-rank 10 pairs (ms),Pipeline 2 total (ms),LLM call P1 (s),LLM call P2 (s)
0,What is the limit on how many Nominated Member...,6.63,28.83,35.46,2.22,2.71
1,How long can the General Secretary speak in th...,7.60,24.12,31.71,2.70,3.80
2,AVERAGE,7.12,26.48,33.58,2.46,3.26


## 8. Scaling: why not run the Cross-Encoder over the whole collection?

In [15]:
# Measure cost per (query, chunk) pair for the cross-encoder vs. per-chunk lookup cost for the bi-encoder.
q = failure_queries[0]["query"]
sample = chunk_texts[:min(200, len(chunk_texts))]
t0 = time.perf_counter(); cross_encoder.predict([(q, c) for c in sample], batch_size=32); t_ce = time.perf_counter() - t0
ce_per_pair = t_ce / len(sample)

t0 = time.perf_counter()
for _ in range(20): baseline_retrieve(q)
bi_per_query = (time.perf_counter() - t0) / 20

print(f"Cross-encoder: {ce_per_pair*1000:.2f} ms per (query, chunk) pair")
print(f"Bi-encoder + FAISS search over {index.ntotal} chunks: {bi_per_query*1000:.2f} ms per query (chunk embeddings precomputed)\n")

scale_rows = []
for n in [len(chunks), 10_000, 100_000, 1_000_000, 10_000_000]:
    scale_rows.append({"Collection size (chunks)": f"{n:,}",
                       "Cross-encoder on ALL chunks, per query (s)": round(ce_per_pair * n, 1),
                       "Cross-encoder on top-10 only (ms)": round(ce_per_pair * 10 * 1000, 1)})
pd.DataFrame(scale_rows)

Cross-encoder: 1.66 ms per (query, chunk) pair
Bi-encoder + FAISS search over 123 chunks: 6.44 ms per query (chunk embeddings precomputed)



,Collection size (chunks),"Cross-encoder on ALL chunks, per query (s)",Cross-encoder on top-10 only (ms)
0,123,0.2,16.6
1,"10,000",16.6,16.6
2,"100,000",165.5,16.6
3,"1,000,000",1655.2,16.6
4,"10,000,000",16551.9,16.6
